In [ ]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

# Fase 1: Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, chi2_contingency
from src.carga import cargar_csv
from src.eda_utils import *

In [ ]:
df = cargar_csv("..\data\\raw\Student_Depression_Dataset_Original.csv")

In [ ]:
# Identificación de tipos de datos.
df.info()

In [ ]:
print(f"Cantidad de objetos en la base de datos: {df.shape[0]} objetos.")
print(f"Cantidad de columnas en la base de datos: {df.shape[1]} columnas.")

# Fase 2a: Análisis exploratorio de datos

In [ ]:
#Estadísticas descriptivas
df.describe()

### Detección de datos atípicos (outliers) en las columnas numéricas

En esta parte, se optó por evaluar las siguientes medidas para la detección de outliers (valores atípicos) dentro del conjunto de datos.

**Método 3 sigma**

Consiste en quedarse con los datos que se encuentran hasta 3 desviaciones estándar alejados de la media.
Tal que:
$$d \in [\bar{x} - 3\sigma, \bar{x} + 3\sigma]$$

**Método IQR**

Consiste en quedarse con los datos están en el rango intercuartil (la diferencia entre el 25% y el 75% del conjunto)
Tal que:
$$d \in [Q1 - 1.5 \cdot IQR, Q3 + 1.5 \cdot IQR]$$


**Asimetría**

Adicionalmente, el coeficiente de asimetría (o skew) servirá para determinar si los datos de una columna se distribuyen normalmente, para eso, se utilza el siguiente cálculo:
$$Skew = \frac{n}{(n-1)(n-2)}\sum_{i=1}^{n}(\frac{x_{i} - \bar{x}}{\sigma})^3$$

Aquellas columnas que tengan un coeficiente de asimetría que esté fuera de los rangos $[-1, 1]$, se consideran columnas en las que los datos NO se distribuyen normalmente.

In [ ]:
outliers_iqr = detectar_outliers_iqr(df)
outliers_3sig = detectar_outliers_3sigmas(df)

In [ ]:
print("***Detección de outliers con método 3 sigmas***", end="\n" + ("-"*40) + "\n")
for col, info in outliers_3sig.items():
    print(f"Columna: {col}")
    print(f"Asimetría (skew): {info['skew']:.3f}")
    print(f"Cantidad de outliers: {info['cantidad_outliers']}")
    print(f"Límite inferior: {info['limite_inferior']:.3f}")
    print(f"Límite superior: {info['limite_superior']:.3f}")
    print("-"*40)

In [ ]:
print("***Detección de outliers con método IQR***", end="\n" + ("-"*40) + "\n")
for col, info in outliers_iqr.items():
    print(f"Columna: {col}")
    print(f"Asimetría (skew): {info['skew']:.3f}")
    print(f"Cantidad de outliers: {info['cantidad_outliers']}")
    print(f"Límite inferior: {info['limite_inferior']:.3f}")
    print(f"Límite superior: {info['limite_superior']:.3f}")
    print("-"*40)

### Conclusiones (outliers)

Pudimos identificar una mayor cantidad de outliers utilizando el método 3 sigma, razón por la cual consideramos utilizar este método por sobre el anterior para ELIMINAR estos outliers.

Si lo queremos ver en cierta perspectiva, en la columna `Age`, por ejemplo, tenemos la presencia de 19 personas que están por sobre los 40 años, sin embargo sus condiciones y estilo de vida (trabajo, familia, responsabilidades) podrían diferir demasiado por sobre el grupo mayoritario (entre 18 y 30 años) para determinar condiciones de depresión. Si quisiéramos obtener conclusiones de por qué un grupo de personas estudiantes mayor a 40 años podría sufrir de depresión, tendríamos que recolectar una mayor muestra de datos, en este caso, optamos más por la eliminación y centrarnos en el estudio del grupo más joven.

Otra observación importante: Tanto `Job Satisfaction` como `Work Pressure` nos presentaron una asimetría exageradamente alta. Al inspeccionar estas columnas, notamos que la mayoría de personas indicó "0" en estos resultados. Según la documentación oficial del dataset, la escala de estas columnas debería ser de 1 a 5, de ser 0, significa que la persona no sabe o no responde. Dada la muy baja participación de personas en estas columnas, consideramos que pierden relevancia tanto en el modelo de ML como en el análisis, así que las eliminaremos directamente.

### Detección de valores nulos
Utilizamos la combinación de `isna()` y `sum()` para esta parte.

In [ ]:
df.isna().sum()

In [ ]:
#Solo una columna, que además se distribuye normalmente
df["Financial Stress"].value_counts()

### Conclusiones (nulos)

`Financial Stress` es la única columna con valores nulos, y solo son 3. Dada la distribución normal de esta columna, decidimos que estos datos van a ser inputados con la mediana (la media puede otorgar decimales, y esta columna solo almacena enteros).

### Detección de datos vacíos no nulos en columnas categóricas

Sin embargo, es importante también analizar las columnas categóricas pues estas podrían tener datos que aunque no sean nulos, no nos otorguen información relevante o sea información vacía.

In [ ]:
#Realizamos una lista de las columnas únicamente categóricas
columnas_categoricas = df.select_dtypes(include=["object", "category"]).columns
columnas_categoricas

In [ ]:
#Recorreremos cada columna categórica y revisaremos sus valores posibles
for i in columnas_categoricas:
    display(df[i].value_counts())

### Conclusiones (categóricas)

Pudimos identificar algunas columnas problemáticas.
En `City`, tenemos unas ciudades mal registradas, algunas están mal escritas, otras son nombres, otras buscan dar direcciones más precisas sin razón ("Less than 5 kalyan") y luego hay otras que directamente no son ciudades, ni son datos correctos ("City", "3.0"). Este punto debe discutirse, puesto que algunas ciudades podrían ser reescritas correctamente, o podría optarse por eliminar aquellas ciudades registradas menos de 5 veces en el dataset.

En `Sleep Duration` y `Dietary Habits`, existen algunos registros donde se indica "Others", cuando los valores de esas categorías ya abarcan todas las respuestas posibles per se, por lo que estos registros con "Others" son datos vacíos y muy propensos a ser eliminados, ya que no podemos asumir la duración del sueño ni el hábito alimenticio de aquellas personas, que por cierto, son un número muy bajo.

En cuanto a `Profession`, podemos notar que hay 27870 personas que respondieron ser estudiantes, dejándonos con un número de personas que ejercen profesión demasiado bajo, tiene sentido si pensamos que en `Job Satisfaction` y `Work Pressure` ya hubo poca participación. Dada esta alta redundancia en esta columna, consideramos que podría ser descartada, pues la información que va a entregar para el modelo de ML va a ser tan poca que perderá relevancia.

